# Day3_10 · 두 집단의 차이 비교

건물 높이 두 집단의 냉난방 부하를 그림, 효과크기, 신뢰구간과 검정 결과로 비교합니다.

## 이 노트북에서 확인할 내용

- 두 집단의 분포와 평균 차이를 확인할 수 있습니다.
- 효과크기와 p-value가 답하는 질문을 구분할 수 있습니다.
- 관찰한 집단 차이를 인과효과로 과장하지 않을 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-10-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-10-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-10-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-10-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-10-05 · pandas와 numpy 불러오기

In [ ]:
import pandas as pd
import numpy as np

### D3B-10-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-10-07 · 건물 에너지자료 읽기

In [ ]:
building = pd.read_csv(DATA_DIR / 'building_energy.csv')

In [ ]:
building.head()

### D3B-10-08 · 건물 높이 종류 확인

In [ ]:
building['overall_height_m'].value_counts().sort_index()

### D3B-10-09 · 높이별 난방 부하 요약

In [ ]:
height_summary = building.groupby('overall_height_m')['heating_load'].agg(['count', 'mean', 'median', 'std'])

In [ ]:
height_summary.round(2)

### D3B-10-10 · 높이별 난방 부하 상자그림

In [ ]:
sns.boxplot(data=building, x='overall_height_m', y='heating_load')
sns.stripplot(data=building, x='overall_height_m', y='heating_load', color='black', alpha=0.18, size=2)
plt.xlabel('Overall height (m)')
plt.ylabel('Heating load')
plt.show()

### D3B-10-11 · 낮은 건물의 난방 부하 선택

In [ ]:
low_height = building.loc[building['overall_height_m'] == 3.5, 'heating_load']

In [ ]:
low_height.head()

### D3B-10-12 · 높은 건물의 난방 부하 선택

In [ ]:
high_height = building.loc[building['overall_height_m'] == 7.0, 'heating_load']

In [ ]:
high_height.head()

### D3B-10-13 · 두 평균의 차이 계산

In [ ]:
mean_difference = high_height.mean() - low_height.mean()

In [ ]:
mean_difference

평균 차이는 실제 단위로 차이의 크기를 보여 줍니다. 다음에는 두 집단 안의 흩어진 정도까지 함께 고려해 차이를 요약합니다.

### D3B-10-14 · 두 집단의 합동 표준편차 계산

In [ ]:
pooled_std = np.sqrt(((len(low_height)-1)*low_height.var(ddof=1) + (len(high_height)-1)*high_height.var(ddof=1)) / (len(low_height)+len(high_height)-2))

In [ ]:
pooled_std

### D3B-10-15 · 표준편차 단위의 차이 계산

In [ ]:
cohens_d = mean_difference / pooled_std

In [ ]:
cohens_d

평균 차이를 집단 내부 표준편차로 나눈 값을 Cohen's d라고 부릅니다. 부호는 방향을, 절댓값은 두 분포가 떨어진 정도를 나타내지만 분야의 실제 단위와 함께 읽어야 합니다.

### D3B-10-16 · 두 집단 평균 비교 함수 불러오기

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu

### D3B-10-17 · Welch t-test 계산

In [ ]:
welch_result = ttest_ind(high_height, low_height, equal_var=False)

In [ ]:
welch_result

### D3B-10-18 · 순위를 이용한 비교 계산

In [ ]:
rank_result = mannwhitneyu(high_height, low_height, alternative='two-sided')

In [ ]:
rank_result

### D3B-10-19 · 두 검정 결과 정리

In [ ]:
test_summary = pd.DataFrame({
    'method': ['Welch t-test', 'Mann-Whitney U'],
    'statistic': [welch_result.statistic, rank_result.statistic],
    'p_value': [welch_result.pvalue, rank_result.pvalue],
})

In [ ]:
test_summary

p-value는 '차이가 없다'는 가정 아래 현재와 같은 결과가 얼마나 드문지를 요약합니다. 차이의 크기나 실무적 중요도를 대신하지 않으므로 평균 차이, 효과크기, 그림과 함께 기록합니다.

### D3B-10-20 · 평균 차이의 bootstrap 값 만들기

In [ ]:
rng = np.random.default_rng(42)
bootstrap_differences = pd.Series([
    rng.choice(high_height, size=len(high_height), replace=True).mean()
    - rng.choice(low_height, size=len(low_height), replace=True).mean()
    for _ in range(2000)
])

In [ ]:
bootstrap_differences.head()

### D3B-10-21 · 평균 차이의 95% 구간 계산

In [ ]:
difference_interval = bootstrap_differences.quantile([0.025, 0.975])

In [ ]:
difference_interval.round(2)

### D3B-10-22 · 평균 차이 분포 그리기

In [ ]:
sns.histplot(bootstrap_differences, bins=35, kde=True)
plt.axvline(0, color='black', linestyle='--')
plt.xlabel('High minus low mean heating load')
plt.show()

### D3B-10-23 · 비교할 결과 열 정하기

In [ ]:
# heating_load를 cooling_load로 바꾸어 보세요.
outcome_column = 'cooling_load'

In [ ]:
outcome_column

### D3B-10-24 · 선택한 결과의 높이별 요약

In [ ]:
chosen_height_summary = building.groupby('overall_height_m')[outcome_column].agg(['count', 'mean', 'median', 'std'])

In [ ]:
chosen_height_summary.round(2)

### D3B-10-25 · 선택한 결과의 상자그림

In [ ]:
sns.boxplot(data=building, x='overall_height_m', y=outcome_column)
plt.title(f'Height groups and {outcome_column}')
plt.show()

### D3B-10-26 · 해석 범위 정리

In [ ]:
comparison_note = pd.Series({
    '관찰한 차이': f'7.0 m 집단과 3.5 m 집단의 {outcome_column} 분포 차이',
    '함께 제시할 값': '집단별 요약, 평균 차이, 효과크기, 신뢰구간, p-value',
    '말할 수 없는 것': '높이 하나만 바꾸었을 때 생기는 인과효과',
})

In [ ]:
comparison_note

### D3B-10-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''building DataFrame과 outcome_column을 사용하세요. 기존 셀은 수정하지 마세요. 두 높이 집단의 평균 차이와 bootstrap 95% 구간을 한 표로 만드는 새 코드 셀을 제안해 주세요. 표 아래에 결과를 인과관계로 단정하지 않는 해석 두 문장을 작성해 주세요.'''.strip()

In [ ]:
codex_request

### D3B-10-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['높이 3.5와 7.0을 비교하는가?', '차이의 방향이 표에 명확한가?', 'p-value만으로 결론내리지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist